# 2. Your first question

Notebook 1 got data onto your screen. This one turns a vague thought into
something data can answer, runs one regression, and reads the output properly.

**Assumed:** you finished notebook 1 and your keys work.

**The honest warning up front.** The hard part is not the regression. It is one
line long. The hard part is deciding what you are asking, and most of this
notebook is that.

## From a thought to a question

Start with something you might actually wonder:

> *Does unemployment have anything to do with inflation?*

That is a thought, not a question. It has no time period, no measure of either
thing, and no statement of what an answer would look like. Sharpen it:

> *Between 1990 and today, in months when US unemployment was higher, was
> inflation lower?*

Now it has a period, two measurable things, and a direction you could confirm
or fail to confirm.

**What this question is not.** It is not "does unemployment CAUSE inflation to
fall". Nothing in this notebook can tell you that, and the gap between *moves
with* and *causes* is most of a degree. Keep the claim the size of the evidence.

In [ ]:
import os
from pathlib import Path

env = Path(".env")
if env.exists():
    for line in env.read_text().splitlines():
        if "=" in line and not line.startswith("#"):
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()

import fred_loader as fl
import pandas as pd

# UNRATE is the unemployment rate. CPIAUCSL is the consumer price index:
# a LEVEL, not a rate of change, which matters in a moment.
raw = fl.pull_fred(["UNRATE", "CPIAUCSL"], start="1990-01-01").dropna()
raw.tail()

## The mistake almost everyone makes here

It is tempting to regress unemployment on `CPIAUCSL` as it stands. Do not.

`CPIAUCSL` is an index that climbs more or less forever. So does almost every
economic level series. Regress one ever-rising thing on another and you get a
confident, beautiful, meaningless result: they both went up over thirty years,
and so did the number of streaming subscriptions.

Inflation is the *rate of change* of that index. That is the thing you actually
meant.

In [ ]:
df = raw.copy()

# Year over year percent change: this month against the same month a year ago.
# 12 because the data is monthly. Comparing to the same month sidesteps
# seasonal effects for free.
df["inflation"] = df["CPIAUCSL"].pct_change(12) * 100
df = df.dropna()

df[["UNRATE", "inflation"]].describe()

Both are now percentages on a comparable scale, and neither trends forever
upward. Look before modelling, same as last time.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["UNRATE"], df["inflation"], s=10, alpha=0.5)
ax.set_xlabel("unemployment rate (%)")
ax.set_ylabel("inflation, year over year (%)")
ax.set_title("Every month since 1991")
plt.show()

**Read that before fitting anything.** Is there a downward tilt? Is it strong or
is it a cloud? Whatever the regression says next should not surprise you after
looking at this. If it does, one of the two is wrong and it is usually not the
picture.

## Set it up with otter

`Pond` is otter's dataset object. You hand it a frame and then *say what role
each column plays*, rather than remembering which is which.

That sounds like bookkeeping. It is the point. Three months later the thing you
will have forgotten is exactly what was on the left of the equation.

In [ ]:
from otter import Pond

pond = Pond(df)
pond.set_dependent("inflation")      # what we are trying to explain
pond.add_independents("UNRATE")      # what we think explains it

# The spec is readable. It is the record of what you asked.
pond.get_spec()

In [ ]:
import statsmodels.api as sm

X = sm.add_constant(pond.get_X())   # the constant is the intercept
y = pond.get_y()

model = sm.OLS(y, X).fit()
print(model.summary())

## Reading the output, line by line

Most of that table is noise until you need it. Four numbers matter today.

**`coef` on UNRATE.** The headline. If it is negative, months with higher
unemployment had lower inflation. The size is in real units: a coefficient of
-0.3 means one percentage point more unemployment goes with 0.3 percentage
points less inflation. Always say the units out loud; a bare "-0.3" is not a
finding.

**`P>|t|`.** Roughly: if there were truly no relationship, how often would you
see a pattern this strong by luck? Small means rarely. **It does not tell you
the relationship is large, or that it matters, or that it is causal.** It is the
single most over-read number in applied work.

**`R-squared`.** How much of the month-to-month variation in inflation this one
variable accounts for. It will be low. That is correct and expected: inflation
has many causes and you included one.

**`No. Observations`.** Check it is the number you expected. If it is far lower,
something quietly dropped rows.

### What you may now say

> Between 1991 and today, US months with higher unemployment tended to have
> lower year-over-year inflation. One variable explains only a small share of
> the variation.

### What you may not say

> Unemployment controls inflation.

You have not ruled out anything else moving both, you have not established
direction, and you have not looked at whether the relationship holds in
different periods. Notebook 3 does the third of those.

## Try it yourself

Change one thing and see what moves:

1. Start at `2010-01-01` instead of 1990. Does the coefficient survive?
2. Add `FEDFUNDS` (the policy interest rate) with `add_controls`. Does the
   unemployment coefficient change? A coefficient that moves a lot when you add
   a control was partly standing in for that control.
3. Drop 2020 and 2021. Two extraordinary years can carry an entire result, and
   finding out whether they do is a normal part of the job, not cheating.

**Next:** [`03-two-sources-one-question.ipynb`](03-two-sources-one-question.ipynb),
where the data comes from two agencies that do not agree about time.